# Getting ChatCPT and Llama to Stream Answers to Technical Questions

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [4]:
openai = OpenAI()

In [5]:
system_role="You are to answer technical questions."

In [6]:
# here is the question; type over this to ask something new

question = """
Please explain what this Python code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [7]:
# Get gpt-4o-mini to answer, with streaming
messages = [
    
    {"role": "system", "content": system_role},
    {"role": "user", "content": question}
]

stream = openai.chat.completions.create(model="gpt-4o-mini", messages=messages,stream=True)
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

The provided Python code is a generator expression that yields values from a set comprehension. Let's break down the code piece by piece:

1. **Set Comprehension**: 
   The expression `{book.get("author") for book in books if book.get("author")}` creates a set of unique authors from a list (or any iterable) called `books`. 
   - `book.get("author")` attempts to retrieve the value associated with the key `"author"` from each `book` dictionary.
   - The `for book in books` iterates through each item in the `books` collection.
   - The `if book.get("author")` condition filters out any books where the "author" key does not exist or is `None`, so only books with a valid author will be included in the resulting set.

2. **Yielding Values**: 
   The `yield from` statement is used to yield all values from the generator produced by the set comprehension. This means that if you were to call this code within a generator function, it would yield each unique author one at a time as they are processed. 

3. **Set Behavior**: 
   By using a set comprehension, any duplicate authors will be eliminated automatically since sets in Python cannot contain duplicate values. In other words, if multiple books have the same author, that author will only appear once in the output.

### Why this is useful:
- This code is likely part of a larger generator function designed to produce a stream of authors from a collection of book dictionaries, without duplicates and while filtering out any entries that do not have an associated author. This can be useful in cases where you need to process or display unique authors from a sizable collection efficiently.

### Example Usage:
Suppose you have a list of books defined as follows:

python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},  # Duplicate author
    {"title": "Book 4"},  # No author
]


Using the code, you would get a set of unique authors:

python
unique_authors = {book.get("author") for book in books if book.get("author")}

# Output: {'Author A', 'Author B'}


If this code were in a generator function, you could iterate over the authors like so:

python
def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

for author in get_unique_authors(books):
    print(author)  # Prints each unique author


This will print `Author A` and `Author B` in no particular order.

In [8]:
# Get Llama 3.2 to answer
OLLAMA_API = "http://localhost:11434/v1"
MODEL = "llama3.2"

In [9]:
# payload = {
#         "model": MODEL,
#         "messages": messages,
#         "stream": True
#     }

In [10]:
from openai import OpenAI
ollama_via_openai = OpenAI(base_url=OLLAMA_API, api_key='ollama')

stream = ollama_via_openai.chat.completions.create(
    model=MODEL,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

**Explaining the Code**

The given Python code is a generator expression that uses the `yields` keyword to produce a sequence of values. The code snippet can be broken down into several components:

1. `yield from`: This keyword is used to delegate iteration to another iterable (in this case, a context manager).
2. `{book.get("author") for book in books if book.get("author")}`: This is a dictionary comprehension followed by another generator expression.

Let's break it down step by step:

- `.get("author")`: This accesses the value associated with the key `"author"` from each map (or dictionary) object (`book`) and returns `None` if the key does not exist.
- `for book in books`: Iterates over each element (`book`) in a list or other iterable of object references, presumably to get the dictionary maps (`books`) themselves.
- `if book.get("author")`: Filters out any elements from `books` where `"author"` is not present as a key. 

However, you can simplify this.

**Example Use Case**

If we have a list of books with their respective authors and the following code:

python
books = [
    {"title": "Book 1", "author": "John"},
    {"title": "Book 2", "country": "USA"}, # does not have 'author'
    {"title": "Book 3", "author": "Alice"}
]

result = [book["author"] for book in books if book.get("author")]


`result` would equal `['John', 'Alice']`, assuming you want to exclude items that don't have an author key.

**Answering the Original Code's Purpose**

The purpose of this code snippet is to extract only those key-value pairs where each value is a valid key within a collection (`books`). Hence, the expression used here is equivalent but has the additional step that we are filtering them away from other items (to ensure all objects have an "author"). 

It effectively reduces `books` into something much more manageable.